# 🎵 WhoDis – Listening Session Pipeline
**Draai deze notebook in [Google Colab](https://colab.research.google.com/)**


In [ ]:
# Installeer gspread (al aanwezig in Colab, maar voor de zekerheid)
!pip install -q gspread

In [ ]:
# ── AUTHENTICATIE ────────────────────────────────────────────────────────────
# Colab gebruikt je ingelogde Google account — geen extra setup nodig.
# Er verschijnt een pop-up om toegang te bevestigen.
from google.colab import auth
auth.authenticate_user()
print('✅ Ingelogd met je Google account.')

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
SPREADSHEET_ID          = '1P0-DbZIPfC4pwHmfDTT93OAOQDggm1ToMdN0UOsoWK4'
WORKSHEET_NAME          = 'check_in'   # exacte tabbladnaam in de sheet
REMINDER_THRESHOLD_DAYS = 3            # na hoeveel dagen iemand flaggen?
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── SHEET INLADEN ────────────────────────────────────────────────────────────
import gspread
import pandas as pd
from google.auth import default

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/spreadsheets.readonly',
    'https://www.googleapis.com/auth/drive.readonly',
])

client = gspread.authorize(creds)
sheet  = client.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)

# get_all_values() ipv get_all_records() omdat de sheet duplicate kolomnamen heeft
# (twee keer 'Welk gevoel had je?' en 'Score van de intensiteit van je gevoel')
all_values = sheet.get_all_values()
headers    = all_values[0]

# Maak kolomnamen uniek door .1 toe te voegen aan duplicaten (zelfde als pandas gedrag)
seen = {}
unique_headers = []
for h in headers:
    if h in seen:
        seen[h] += 1
        unique_headers.append(f'{h}.{seen[h]}')
    else:
        seen[h] = 0
        unique_headers.append(h)

df = pd.DataFrame(all_values[1:], columns=unique_headers)

print(f'✔  {len(df)} rijen geladen uit "{WORKSHEET_NAME}"')
df.head(3)

In [ ]:
# ── KOLOMMEN HERNOEMEN & TYPES INSTELLEN ─────────────────────────────────────
COLUMN_MAP = {
    'Tijdstempel'                                    : 'timestamp',
    'Deelnemerscode'                                 : 'participant',
    'Welke playlist luisterde je?'                   : 'playlist',
    'Welke dag deed je een check-in?'                : 'checkin_date',
    'Starttijd?'                                     : 'start_time',
    'Eindtijd?'                                      : 'end_time',
    'Welk gevoel had je?'                            : 'mood_before',
    'Score van de intensiteit van je gevoel'         : 'mood_score_before',
    'Welk gevoel had je?.1'                          : 'mood_after',
    'Score van de intensiteit van je gevoel.1'       : 'mood_score_after',
    'Waar heb je geluisterd?'                        : 'location',
    'Heb je de volledige playlist kunnen luisteren?' : 'full_playlist',
}

df = df.rename(columns=COLUMN_MAP)
df['participant_normalized'] = df['participant'].str.replace(' ', '').str.lower()
df['timestamp']         = pd.to_datetime(df['timestamp'], dayfirst=True, errors='coerce')
df['mood_score_before'] = pd.to_numeric(df.get('mood_score_before'), errors='coerce')
df['mood_score_after']  = pd.to_numeric(df.get('mood_score_after'),  errors='coerce')

print('Kolomtypes:')
df.dtypes

In [ ]:
# ── LAATSTE SESSIE PER DEELNEMER ─────────────────────────────────────────────
vandaag = pd.Timestamp.today().normalize()

laatste = (
    df.groupby('participant_normalized')['timestamp']
      .max()
      .reset_index()
)
laatste['days_since'] = (vandaag - laatste['timestamp']).dt.days.clip(lower=0)
laatste = laatste.sort_values('days_since', ascending=False).reset_index(drop=True)

print(f'📅 Status op {vandaag.date()}\n')
laatste

In [ ]:
# ── WIE HEEFT EEN REMINDER NODIG? ────────────────────────────────────────────
needs_reminder = laatste[laatste['days_since'] >= REMINDER_THRESHOLD_DAYS]

if needs_reminder.empty:
    print('✅ Iedereen heeft recentelijk ingecheckt — geen reminders nodig!')
else:
    print(f'⚠️  {len(needs_reminder)} deelnemer(s) hebben al {REMINDER_THRESHOLD_DAYS}+ dagen niet ingecheckt:\n')
    for _, row in needs_reminder.iterrows():
        print(f"  • {row['participant']:15s}  laatste sessie: {row['timestamp'].date()}  ({row['days_since']} dagen geleden)")

In [ ]:
# ── OVERZICHT PER DEELNEMER ──────────────────────────────────────────────────
summary = (
    df.groupby('participant_normalized')
      .agg(
          totaal_sessies  = ('timestamp', 'count'),
          playlists       = ('playlist', lambda x: ', '.join(sorted(x.dropna().unique()))),
          gem_mood_delta  = ('mood_score_after',
                             lambda x: round(
                                 (x - df.loc[x.index, 'mood_score_before']).mean(), 2
                             ))
      )
      .reset_index()
      .merge(laatste[['participant_normalized', 'days_since']], on='participant_normalized')
      .sort_values('days_since', ascending=False)
)

print('📊 Overzicht deelnemers')
summary